# CS 562 — Step 3: Reasoning-Quality Verification

**Cell 1** — Configuration (edit model, paths, prompt here)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 CONFIGURATION — edit this cell to swap models / tweak settings ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os

# ── API Setup ──────────────────────────────────────────────────────────────
TOGETHER_API_KEY = os.environ.get("TOGETHER_API_KEY", "") # <-- Set this in your environment variables
if not TOGETHER_API_KEY:
    raise ValueError("TOGETHER_API_KEY environment variable not set. Please set it to your API key.")

# ── Model Selection ────────────────────────────────────────────────────────
MODEL = "Qwen/Qwen3-235B-A22B-Thinking-2507"

# Set True for models that emit <think>...</think> blocks

IS_THINKING_MODEL = True

# ── File Paths ─────────────────────────────────────────────────────────────
INPUT_FILE    = "merged_results.json"
VERIFIED_FILE = "step3_verified.json"
REJECTED_FILE = "step3_rejected.json"

# ── Testing / Sampling ─────────────────────────────────────────────────────
MAX_SAMPLES = None

# ── Language Columns ───────────────────────────────────────────────────────
# Now includes all three languages: English, Chinese, Spanish.
LANGUAGES = {
    "en": {"question": "question_en", "options": "options_en", "reasoning": "reasoning_en", "prediction": "prediction_en"},
    "zh": {"question": "question_zh", "options": "options_zh", "reasoning": "reasoning_zh", "prediction": "prediction_zh"},
    "es": {"question": "question_es", "options": "options_es", "reasoning": "reasoning_es", "prediction": "prediction_es"},
}

# ── Inference Settings ─────────────────────────────────────────────────────
CONCURRENCY = 10
TEMPERATURE = 0.0
MAX_TOKENS  = 1024
MAX_RETRIES = 3

# ── Evaluation Prompt ──────────────────────────────────────────────────────
SYSTEM_PROMPT = """\
You are a rigorous reasoning-verification expert. You will receive:
- A multiple-choice question with answer options
- The confirmed correct answer letter
- A model-generated reasoning trace

The final answer is already known to be CORRECT. Your ONLY job is to judge \
whether the REASONING STEPS are logically valid.

IMPORTANT: The reasoning trace may be TRUNCATED (cut off mid-sentence). \
This is expected. Judge only the steps that ARE present:
- If the visible steps are all logically sound, verdict is VALID.
- If any visible step contains an error, verdict is INVALID.
- Do NOT mark a trace as INVALID solely because it is incomplete.

A reasoning trace is VALID if:
- Each visible step follows logically from the problem or previous steps.
- There are no mathematical, logical, or factual errors in any visible step.
- The reasoning (as far as it goes) genuinely supports the correct answer \
  rather than reaching it by luck or flawed logic.

A reasoning trace is INVALID if:
- Any step contains a computational or logical error.
- The reasoning is circular, contradictory, or incoherent.
- The trace misidentifies options or concepts even if the final answer is correct.
- The steps that are present argue toward a WRONG answer (even though the \
  recorded final answer is correct).

The question and reasoning trace may be in any language. Evaluate the logical \
validity regardless of language.

Respond with ONLY a JSON object in this exact format (no other text):
{"verdict": "VALID" or "INVALID", "issues": ["specific issues if INVALID, empty list if VALID"]}
"""

print(f"Config loaded.")
print(f"  Model        : {MODEL}")
print(f"  Thinking     : {IS_THINKING_MODEL}")
print(f"  Input        : {INPUT_FILE}")
print(f"  Max samples  : {MAX_SAMPLES or 'ALL'}")
print(f"  Languages    : {list(LANGUAGES.keys())}")
print(f"  Concurrency  : {CONCURRENCY}")


Config loaded.
  Model        : Qwen/Qwen3-235B-A22B-Thinking-2507
  Thinking     : True
  Input        : step2_filtered.json
  Max samples  : 3
  Languages    : ['en', 'zh']
  Concurrency  : 10


**Cell 2** — Pipeline (run as-is)

In [2]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 3 PIPELINE — run this cell after setting config above            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import asyncio
import json
import re
import time
from together import AsyncTogether

# ── Helpers ────────────────────────────────────────────────────────────────

USER_TEMPLATE = """\
Question:
{question}

Options:
{options}

Correct answer: {gold_answer}

--- MODEL REASONING TRACE (may be truncated) ---
{trace}
--- END OF TRACE ---

Judge whether the reasoning steps shown above are logically valid.
"""

def format_options(options) -> str:
    """Format options for the evaluator prompt.
    
    Handles both formats:
      - dict:  {"A": "text", "B": "text", ...}
      - list:  ["text0", "text1", ...]
    """
    if isinstance(options, dict):
        return "\n".join(f"{k}. {v}" for k, v in sorted(options.items()))
    # Legacy list format
    labels = "ABCDEFGHIJ"
    return "\n".join(f"{labels[i]}. {opt}" for i, opt in enumerate(options))


def extract_json_verdict(content: str, is_thinking: bool) -> dict:
    """Parse the evaluator JSON verdict from model output."""
    if is_thinking:
        content = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL)
    content = content.strip()
    if content.startswith("```"):
        content = re.sub(r"^```(?:json)?\n?", "", content)
        content = re.sub(r"\n?```$", "", content)
    match = re.search(r"\{.*\}", content, re.DOTALL)
    if not match:
        raise json.JSONDecodeError("No JSON object found", content, 0)
    parsed = json.loads(match.group())
    verdict = parsed.get("verdict", "").strip().upper()
    if verdict not in ("VALID", "INVALID"):
        raise ValueError(f"Unexpected verdict: {verdict}")
    parsed["verdict"] = verdict
    return parsed


async def evaluate_trace(client, semaphore, qid, lang, question, options, gold_answer, trace):
    """Evaluate one (question, language) reasoning trace."""
    user_msg = USER_TEMPLATE.format(
        question=question,
        options=format_options(options),
        gold_answer=gold_answer,
        trace=trace,
    )
    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = await client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_msg},
                    ],
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                )
                content = response.choices[0].message.content
                verdict = extract_json_verdict(content, IS_THINKING_MODEL)
                return {
                    "question_id": qid,
                    "lang":        lang,
                    "verdict":     verdict["verdict"],
                    "issues":      verdict.get("issues", []),
                }
            except Exception as exc:
                if attempt == MAX_RETRIES:
                    return {
                        "question_id": qid,
                        "lang":        lang,
                        "verdict":     "ERROR",
                        "issues":      [f"{type(exc).__name__}: {exc}"],
                    }
                await asyncio.sleep(2 * attempt)

    return {"question_id": qid, "lang": lang, "verdict": "ERROR", "issues": ["Exhausted retries"]}


# ── Deduplication & correct-language detection ─────────────────────────────

def deduplicate_and_tag(raw_samples):
    """Deduplicate merged_results.json and determine which languages are
    correct for each unique question.
    
    The merged file contains entries from three pair groups (en–zh, en–es,
    zh–es). A question may appear up to 3 times if all three pairs had
    matching correct answers. We deduplicate by question_id and build a
    set of languages whose prediction matches gold for that question.
    
    Returns:
        list of (entry_dict, correct_langs_set) tuples, one per unique question.
    """
    seen = {}  # question_id -> (entry, set_of_correct_langs)
    for entry in raw_samples:
        qid  = entry["question_id"]
        gold = entry["gold_answer"]

        # Determine which languages are correct in this row
        correct_here = set()
        for lang, fields in LANGUAGES.items():
            if entry[fields["prediction"]] == gold:
                correct_here.add(lang)

        if qid not in seen:
            seen[qid] = (entry, correct_here)
        else:
            # Merge correct languages from duplicate rows
            seen[qid] = (seen[qid][0], seen[qid][1] | correct_here)

    return list(seen.values())


# ── Main pipeline ──────────────────────────────────────────────────────────

async def run_pipeline():
    with open(INPUT_FILE, encoding="utf-8") as f:
        raw_samples = json.load(f)

    print(f"Loaded {len(raw_samples)} rows from {INPUT_FILE}")

    # Deduplicate
    deduped = deduplicate_and_tag(raw_samples)
    print(f"After deduplication: {len(deduped)} unique questions")

    # Apply MAX_SAMPLES limit
    if MAX_SAMPLES is not None:
        deduped = deduped[:MAX_SAMPLES]
        print(f"Using first {len(deduped)} (MAX_SAMPLES={MAX_SAMPLES})")

    client = AsyncTogether(api_key=TOGETHER_API_KEY)
    semaphore = asyncio.Semaphore(CONCURRENCY)

    # Build one task per (question, correct_language) pair
    # Only evaluate languages whose prediction matched gold.
    tasks = []
    task_meta = []  # parallel list: (qid, correct_langs)
    for entry, correct_langs in deduped:
        qid  = entry["question_id"]
        gold = entry["gold_answer"]
        for lang, fields in LANGUAGES.items():
            if lang not in correct_langs:
                continue  # skip languages that got the wrong answer
            question = entry[fields["question"]]
            options  = entry[fields["options"]]
            trace    = entry[fields["reasoning"]]
            tasks.append(
                evaluate_trace(client, semaphore, qid, lang, question, options, gold, trace)
            )

    total = len(tasks)
    print(f"Evaluating {total} traces ({len(deduped)} questions, only correct-answer languages)…\n")

    results = []
    done = 0
    for coro in asyncio.as_completed(tasks):
        result = await coro
        results.append(result)
        done += 1
        tag = "✓" if result["verdict"] == "VALID" else ("✗" if result["verdict"] == "INVALID" else "?")
        print(f"  [{done:>{len(str(total))}}/{total}] q{result['question_id']}/{result['lang']} {tag}")

    # ── Group verdicts by question_id ──────────────────────────────────
    verdicts_by_q = {}
    for r in results:
        verdicts_by_q.setdefault(r["question_id"], []).append(r)

    verified = []
    rejected = []

    for entry, correct_langs in deduped:
        qid = entry["question_id"]
        q_verdicts = verdicts_by_q.get(qid, [])

        eval_results = {
            v["lang"]: {"verdict": v["verdict"], "issues": v["issues"]}
            for v in q_verdicts
        }

        # A question is verified only if ALL evaluated languages returned VALID
        all_valid = (
            all(v["verdict"] == "VALID" for v in q_verdicts)
            and len(q_verdicts) == len(correct_langs)
            and len(q_verdicts) > 0
        )

        # Annotate with which languages were correct (for downstream reference)
        enriched = {**entry, "correct_langs": sorted(correct_langs)}

        if all_valid:
            verified.append(enriched)
        else:
            rejected.append({**enriched, "eval_results": eval_results})

    # ── Write outputs ──────────────────────────────────────────────────
    with open(VERIFIED_FILE, "w", encoding="utf-8") as f:
        json.dump(verified, f, ensure_ascii=False, indent=2)
    with open(REJECTED_FILE, "w", encoding="utf-8") as f:
        json.dump(rejected, f, ensure_ascii=False, indent=2)

    # ── Summary ────────────────────────────────────────────────────────
    n_valid   = sum(1 for r in results if r["verdict"] == "VALID")
    n_invalid = sum(1 for r in results if r["verdict"] == "INVALID")
    n_error   = sum(1 for r in results if r["verdict"] == "ERROR")

    print(f"\n{'='*55}")
    print(f" TRACE-LEVEL RESULTS")
    print(f"   Total traces : {total}")
    print(f"   Valid        : {n_valid}")
    print(f"   Invalid      : {n_invalid}")
    print(f"   Errors       : {n_error}")
    print(f"\n QUESTION-LEVEL RESULTS")
    print(f"   Total        : {len(deduped)}")
    print(f"   Verified     : {len(verified)}")
    print(f"   Rejected     : {len(rejected)}")
    print(f"\n Outputs:")
    print(f"   {VERIFIED_FILE}  ({len(verified)} questions)")
    print(f"   {REJECTED_FILE}  ({len(rejected)} questions)")
    print(f"{'='*55}")

    await client.close()

t0 = time.time()
await run_pipeline()
print(f"\nDone in {time.time() - t0:.1f}s")


Loaded 9 questions, using first 3 (MAX_SAMPLES=3)
Evaluating 6 traces (3 questions × 2 languages)…

  [1/6] q224/zh ✓
  [2/6] q202/zh ✓
  [3/6] q414/zh ✓
  [4/6] q202/en ✓
  [5/6] q224/en ✓
  [6/6] q414/en ✓

 TRACE-LEVEL RESULTS
   Total traces : 6
   Valid        : 6
   Invalid      : 0
   Errors       : 0

 QUESTION-LEVEL RESULTS
   Total        : 3
   Verified     : 3
   Rejected     : 0

 Outputs:
   step3_verified.json  (3 questions)
   step3_rejected.json  (0 questions)

Done in 29.1s


**Cell 3** — Inspect results

In [3]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  INSPECT RESULTS — run after pipeline finishes                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import json

print("=" * 55)
print(" VERIFIED SAMPLES")
print("=" * 55)
with open(VERIFIED_FILE, encoding="utf-8") as f:
    verified = json.load(f)
print(f"Total verified: {len(verified)}")
for entry in verified[:5]:
    langs = ", ".join(entry.get("correct_langs", []))
    print(f"  q{entry['question_id']} (gold: {entry['gold_answer']}, correct langs: {langs})")

print(f"\n{'='*55}")
print(" REJECTED SAMPLES (with evaluator feedback)")
print("=" * 55)
with open(REJECTED_FILE, encoding="utf-8") as f:
    rejected = json.load(f)
print(f"Total rejected: {len(rejected)}\n")

for entry in rejected[:5]:
    langs = ", ".join(entry.get("correct_langs", []))
    print(f"Question {entry['question_id']} (gold: {entry['gold_answer']}, correct langs: {langs}):")
    if "eval_results" in entry:
        for lang, ev in entry["eval_results"].items():
            status = "✓ VALID" if ev["verdict"] == "VALID" else "✗ " + ev["verdict"]
            print(f"  [{lang}] {status}")
            for issue in ev.get("issues", []):
                print(f"        → {issue}")
    print()


 VERIFIED SAMPLES
Total verified: 2
  q202 (gold: B)
  q414 (gold: B)

 REJECTED SAMPLES (with evaluator feedback)
Total rejected: 1

Question 224 (gold: I):
  [zh] ✗ INVALID
        → The reasoning explicitly states that option J is incorrect ("J. 行为模式完全由遗传决定 - 这个说法不正确"), but later claims the answer is 'most consistent with I and J', creating a direct logical contradiction in the visible steps. This invalidates the reasoning process despite the correct final selection of I.
  [en] ✓ VALID



---
## Step 4: Reasoning Divergence Analysis

Measures cosine similarity between reasoning traces across language pairs using a multilingual sentence embedding model (LaBSE).

Only compares languages whose predictions matched the gold answer **and** whose reasoning was verified VALID in the cells above.

$$D_{\text{reason}}(q_i) = \mathbb{E}_{k,k'} \left[ d(r_{i,k}, r_{i,k'}) \right]$$

where $d$ is cosine distance $(1 - \text{cosine\_similarity})$.

**Cell 4** — Divergence Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 4 CONFIGURATION — edit embedding model / batch size here          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Embedding Model ───────────────────────────────────────────────────────
EMBED_MODEL_NAME = "sentence-transformers/LaBSE"   # strong cross-lingual model
EMBED_BATCH_SIZE = 32

# LaBSE does not need a task prefix; set True if switching to E5 models.
USE_PREFIX = False

# ── File Paths ─────────────────────────────────────────────────────────────
DIVERGENCE_INPUT  = VERIFIED_FILE          # re-use the path from Step 3 config
DIVERGENCE_OUTPUT = "step4_reasoning_divergence.json"

print(f"Divergence config loaded.")
print(f"  Embedding model : {EMBED_MODEL_NAME}")
print(f"  Batch size      : {EMBED_BATCH_SIZE}")
print(f"  Input           : {DIVERGENCE_INPUT}")
print(f"  Output          : {DIVERGENCE_OUTPUT}")
print(f"  Use prefix      : {USE_PREFIX}")

**Cell 5** — Divergence Pipeline (run as-is)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STEP 4 PIPELINE — reasoning divergence via cross-lingual embeddings    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import json
import torch
import numpy as np
from itertools import combinations
from sentence_transformers import SentenceTransformer
import pandas as pd


def encode_traces(model, texts, prompt_prefix=""):
    """Encode a list of texts with optional task prefix."""
    prefixed = [f"{prompt_prefix}{t}" for t in texts]
    return model.encode(
        prefixed,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,   # L2-normalise → dot product == cosine sim
        convert_to_numpy=True,
    )


# ── Load verified data ────────────────────────────────────────────────────
print("Loading verified data …")
with open(DIVERGENCE_INPUT, "r", encoding="utf-8") as f:
    records = json.load(f)
print(f"  {len(records)} questions loaded.")

# Filter: need ≥2 correct languages to form at least one pair
valid = [r for r in records if len(r.get("correct_langs", [])) >= 2]
print(f"  {len(valid)} records with ≥2 correct languages.")

# ── Load embedding model ──────────────────────────────────────────────────
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"\nLoading model: {EMBED_MODEL_NAME}  (device: {device})")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)

prefix = "query: " if USE_PREFIX else ""

# ── Collect all unique traces to encode ───────────────────────────────────
trace_texts = []
trace_index = {}   # (record_idx, lang) → flat index
for rec_idx, r in enumerate(valid):
    for lang in r["correct_langs"]:
        key = (rec_idx, lang)
        trace_index[key] = len(trace_texts)
        trace_texts.append(r[f"reasoning_{lang}"])

print(f"\nEncoding {len(trace_texts)} reasoning traces …")
embeddings = encode_traces(embed_model, trace_texts, prompt_prefix=prefix)

# ── Compute pairwise similarities ─────────────────────────────────────────
results = []       # one entry per question (averaged across pairs)
pair_results = []  # one entry per (question, lang_pair)

for rec_idx, r in enumerate(valid):
    langs = r["correct_langs"]
    pairs = list(combinations(sorted(langs), 2))
    pair_sims = {}
    for lang_a, lang_b in pairs:
        emb_a = embeddings[trace_index[(rec_idx, lang_a)]]
        emb_b = embeddings[trace_index[(rec_idx, lang_b)]]
        sim = float(np.dot(emb_a, emb_b))  # already L2-normalised
        pair_key = f"{lang_a}-{lang_b}"
        pair_sims[pair_key] = sim
        pair_results.append({
            "question_id":      r["question_id"],
            "pair":             pair_key,
            "cosine_similarity": sim,
            "cosine_distance":   1.0 - sim,
        })

    # D_reason for this question: average cosine distance across all valid pairs
    mean_sim  = float(np.mean(list(pair_sims.values())))
    mean_dist = 1.0 - mean_sim
    results.append({
        "question_id":       r["question_id"],
        "correct_langs":     langs,
        "pair_similarities": pair_sims,
        "cosine_similarity": mean_sim,
        "cosine_distance":   mean_dist,   # D_reason per question
    })

# ── Summary statistics ────────────────────────────────────────────────────
df = pd.DataFrame(results)
df_pairs = pd.DataFrame(pair_results)

print("\n── Reasoning Divergence Summary (question-level, averaged) ──────")
print(f"  N questions             : {len(df)}")
print(f"  Mean cosine similarity  : {df['cosine_similarity'].mean():.4f}")
print(f"  Std  cosine similarity  : {df['cosine_similarity'].std():.4f}")
print(f"  Min  cosine similarity  : {df['cosine_similarity'].min():.4f}")
print(f"  Max  cosine similarity  : {df['cosine_similarity'].max():.4f}")
print(f"  Mean D_reason           : {df['cosine_distance'].mean():.4f}")
print("──────────────────────────────────────────────────────────────────")

# Per language-pair breakdown
print("\n── Per Language-Pair Breakdown ───────────────────────────────────")
for pair_name, grp in df_pairs.groupby("pair"):
    print(f"  {pair_name}:  N={len(grp)},  mean_sim={grp['cosine_similarity'].mean():.4f},  "
          f"std={grp['cosine_similarity'].std():.4f}")
print("──────────────────────────────────────────────────────────────────")

# Top-5 most / least similar
top5_similar  = df.nlargest(5, "cosine_similarity")[["question_id", "cosine_similarity"]]
top5_diverged = df.nlargest(5, "cosine_distance")[["question_id", "cosine_distance"]]
print("\nTop-5 most similar reasoning (question-level avg):")
print(top5_similar.to_string(index=False))
print("\nTop-5 most divergent reasoning (question-level avg):")
print(top5_diverged.to_string(index=False))

# ── Save ─────────────────────────────────────────────────────────────────
pair_summary = {}
for pair_name, grp in df_pairs.groupby("pair"):
    pair_summary[pair_name] = {
        "n":                      len(grp),
        "mean_cosine_similarity": float(grp["cosine_similarity"].mean()),
        "std_cosine_similarity":  float(grp["cosine_similarity"].std()),
    }

output = {
    "summary": {
        "n":                        len(df),
        "mean_cosine_similarity":   float(df["cosine_similarity"].mean()),
        "std_cosine_similarity":    float(df["cosine_similarity"].std()),
        "min_cosine_similarity":    float(df["cosine_similarity"].min()),
        "max_cosine_similarity":    float(df["cosine_similarity"].max()),
        "mean_reasoning_divergence": float(df["cosine_distance"].mean()),
    },
    "per_pair_summary": pair_summary,
    "per_question": results,
}
with open(DIVERGENCE_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print(f"\nResults saved to {DIVERGENCE_OUTPUT}")

**Cell 6** — Inspect Divergence Results

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  INSPECT DIVERGENCE — quick look at the saved output                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import json
import pandas as pd

with open(DIVERGENCE_OUTPUT, "r", encoding="utf-8") as f:
    div_data = json.load(f)

print("=" * 55)
print(" DIVERGENCE SUMMARY")
print("=" * 55)
for k, v in div_data["summary"].items():
    print(f"  {k:30s}: {v}")

print(f"\n{'='*55}")
print(" PER-PAIR SUMMARY")
print("=" * 55)
for pair, stats in div_data["per_pair_summary"].items():
    print(f"  {pair}: N={stats['n']}, mean_sim={stats['mean_cosine_similarity']:.4f}, "
          f"std={stats['std_cosine_similarity']:.4f}")

# Distribution plot (if matplotlib available)
try:
    import matplotlib.pyplot as plt

    df_q = pd.DataFrame(div_data["per_question"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram of question-level cosine similarity
    axes[0].hist(df_q["cosine_similarity"], bins=30, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Cosine Similarity (question avg)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Distribution of Cross-Lingual Reasoning Similarity")
    axes[0].axvline(df_q["cosine_similarity"].mean(), color="red", linestyle="--",
                    label=f"mean={df_q['cosine_similarity'].mean():.3f}")
    axes[0].legend()

    # Per-pair box plot
    pair_data = []
    for entry in div_data["per_question"]:
        for pair, sim in entry["pair_similarities"].items():
            pair_data.append({"pair": pair, "cosine_similarity": sim})
    df_p = pd.DataFrame(pair_data)
    df_p.boxplot(column="cosine_similarity", by="pair", ax=axes[1])
    axes[1].set_title("Cosine Similarity by Language Pair")
    axes[1].set_xlabel("Language Pair")
    axes[1].set_ylabel("Cosine Similarity")
    plt.suptitle("")  # remove auto-title from boxplot

    plt.tight_layout()
    plt.show()
except ImportError:
    print("\n(matplotlib not available — skipping plots)")